В корпусе отсутствовали аннотации для статей 2002, 2004-2006, 2008 и 2011 годов

In [ ]:
# Импортируем необходимые библиотеки
from pathlib import Path  # для работы с путями файлов
from PyPDF2 import PdfReader  # для чтения PDF файлов
import os  # для работы с операционной системой

def extract_text_from_pdfs(pdf_folder="arxiv_papers"):
    """
    Функция для извлечения текста из PDF файлов
    """
    corpus = []  # создаем пустой список для хранения текстов
    
    # Проходим по всем PDF файлам в указанной папке
    for pdf_file in Path(pdf_folder).glob("*.pdf"):
        try:
            # Создаем объект для чтения PDF
            reader = PdfReader(str(pdf_file))
            # Извлекаем текст из каждой страницы и объединяем их
            text = "\n".join(page.extract_text() for page in reader.pages if page.extract_text())
            corpus.append(text)  # добавляем текст в корпус
        except Exception as e:
            # В случае ошибки выводим сообщение
            print(f"Ошибка в {pdf_file.name}: {e}")
    return corpus

# Вызываем функцию для извлечения текстов
corpus_texts = extract_text_from_pdfs()



In [ ]:
texts = []

In [ ]:
# Проходим по каждому тексту в корпусе
for text in corpus_texts:
    # Создаем уникальное имя для каждого документа в формате "paper_X"
    name = "paper_" + str(len(texts))
    # Устанавливаем год публикации (в данном случае фиксированный 2025)
    year = "2025"  
    # Добавляем кортеж (имя, текст, год) в список texts
    texts.append((name, text, year))

In [ ]:
texts[:1]

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(texts, columns=['names', 'texts', 'years'])

df

## Sumy

In [ ]:
pip install sumy

In [ ]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

import nltk
nltk.download('punkt')

In [ ]:
summarizer = LexRankSummarizer()

In [ ]:
abstracts = []

In [ ]:
# Проходим по каждому тексту в датафрейме
for text in df.texts:
  # Сохраняем текст в переменную doc
  doc = text
  # Создаем парсер для русского текста
  parser = PlaintextParser.from_string(doc, Tokenizer('russian'))
  # Генерируем краткое содержание из 3 предложений
  summary = summarizer(parser.document, 3)
  # Добавляем полученные предложения в список abstracts
  abstracts.append([str(sentence) for sentence in summary])

In [ ]:
abstracts[:10]

In [ ]:
annotations = []

In [ ]:
for abstr in enumerate(abstracts, start=1):
  print(f'Аннотация статьи {abstr[0]}:')
  annotation = str(' '.join(abstr[1]))
  annotations.append(annotation)
  print(annotation, '\n\n')

In [ ]:
annots = pd.DataFrame(annotations, columns=['annotations_sumy'])

In [ ]:
sumy = pd.concat([df, annots], join='outer', axis=1)

In [ ]:
sumy

##T5

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoModelForSeq2SeqLM, T5TokenizerFast

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelWithLMHead, T5ForConditionalGeneration

##RuT5-base-sum

In [ ]:
model_name = "IlyaGusev/rut5_base_sum_gazeta"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

In [ ]:
# Цикл по первым 5 текстам из датафрейма
for text in df.texts[:5]:
  # Сохраняем текст статьи
  article_text = text
  
  # Токенизируем текст с ограничением длины 600 токенов
  # add_special_tokens=True - добавляет специальные токены [CLS] (начало последовательности) и [SEP] (разделитель последовательностей)
  # padding="max_length" - дополняет последовательности до максимальной длины
  # truncation=True - обрезает текст если он длиннее max_length
  # return_tensors="pt" - возвращает тензоры PyTorch
  input_ids = tokenizer([article_text], max_length=600, add_special_tokens=True, padding="max_length", truncation=True, return_tensors="pt")["input_ids"]
  
  # Генерируем аннотацию с помощью модели
  # no_repeat_ngram_size=4 - запрещает повторение n-грамм длиной 4
  output_ids = model.generate(input_ids=input_ids, no_repeat_ngram_size=4)[0]
  
  # Декодируем полученные токены обратно в текст
  summary = tokenizer.decode(output_ids, skip_special_tokens=True)
  
  # Выводим аннотацию
  print(f'Аннотация текста:\n {summary}\n\n')

In [ ]:
# Создаем пустой список для хранения аннотаций
annotations_t5 = []

# Проходим по всем текстам в датафрейме
for text in df.texts:
  # Сохраняем текст статьи
  article_text = text
  
  # Токенизируем текст с ограничением длины 600 токенов
  # add_special_tokens=True - добавляет специальные токены [CLS] и [SEP]
  # padding="max_length" - дополняет последовательности до максимальной длины
  # truncation=True - обрезает текст если он длиннее max_length
  # return_tensors="pt" - возвращает тензоры PyTorch
  input_ids = tokenizer([article_text], max_length=600, add_special_tokens=True, padding="max_length", truncation=True, return_tensors="pt")["input_ids"]
  
  # Генерируем аннотацию с помощью модели
  # min_length=50 - минимальная длина аннотации
  # max_length=250 - максимальная длина аннотации
  # no_repeat_ngram_size=4 - запрещает повторение n-грамм длиной 4
  output_ids = model.generate(input_ids=input_ids, min_length=50, max_length=250, no_repeat_ngram_size=4)[0]
  
  # Декодируем полученные токены обратно в текст
  summary = tokenizer.decode(output_ids, skip_special_tokens=True)
  
  # Выводим аннотацию
  print(f'Аннотация текста:\n {summary}\n\n')
  
  # Добавляем аннотацию в список
  annotations_t5.append(str(''.join(summary)))

In [ ]:
annots_t5 = pd.DataFrame(annotations_t5, columns=['annotations_t5'])

In [ ]:
full = pd.concat([sumy, annots_t5], join='outer', axis=1)

In [ ]:
full

In [ ]:
full.to_csv('summarization.csv')

In [ ]:
for index, row in full.iterrows():
    year = row['years']
    name = row['names']
    text = row['annotations_t5']
    
    os.makedirs(f'{year}_annotations', exist_ok=True)
    
    filepath = os.path.join(f'{year}_annotations', f'{name}_annotation.txt')
    with open(filepath, 'w', encoding='utf-8') as writefile:
        writefile.write(text)